# 03 실전 반도체 공정 데이터 분석 · 복습 빈칸 채우기 — 학생용

수업에서 배운 핵심 코드를 빈칸을 채우며 다시 익혀 봅니다.

- 코드 안의 `___`를 알맞은 코드로 바꾼 뒤 셀을 실행하세요.
- 막히면 **힌트**를, 실행한 뒤에는 **예상 결과**를 열어 내 결과와 비교하세요.
- 위에서 아래로 순서대로 실행하세요. 앞 셀에서 만든 변수를 뒤 셀에서 사용하므로, 앞 빈칸을 채워야 다음 셀이 실행됩니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.

### 준비 · 라이브러리 불러오기

빈칸이 없는 셀입니다. 먼저 실행하세요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             accuracy_score, recall_score)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1~2단계 · 데이터 불러오기 & 탐색

컬럼이 590개가 넘으므로 요약 정보로 살핍니다.

### Q1. 데이터 읽고 크기 확인하기

fab.csv를 읽고 행·열 수를 출력합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

CSV 읽기는 `read_csv()`, (행, 열) 크기는 `shape`입니다.

</details>

In [ ]:
df = pd.___('fab.csv')
print(f"데이터 크기: {df.___[0]}행 × {df.shape[1]}열")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
데이터 크기: 1567행 × 592열
```

</details>

### Q2. 열 목록 없이 요약 보기

열이 너무 많으니 열별 목록은 빼고 요약만 봅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

'자세히(장황하게)'라는 뜻의 옵션을 False로 둡니다.

</details>

In [ ]:
df.info(___=False)

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, SensorTime to Pass_Fail
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB
```

</details>

### Q3. 라벨 분포 확인하기

정상(-1)과 불량(+1)이 각각 몇 건인지 봅니다. 불균형이 얼마나 심한지 확인하세요.

<details>
<summary><strong>💡 힌트</strong></summary>

값별 개수를 세는 함수입니다.

</details>

In [ ]:
df['Pass_Fail'].___()

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
Pass_Fail
-1    1463
 1     104
Name: count, dtype: int64
```

</details>

## 3~4단계 · 결측값 처리 · 의미 없는 컬럼 제거

### Q4. 컬럼별 결측률 계산하기

결측값 개수를 전체 행 수로 나눠 %로 만듭니다.

<details>
<summary><strong>💡 힌트</strong></summary>

결측 여부 함수와, 행 수를 세는 파이썬 기본 함수입니다.

</details>

In [ ]:
miss_pct = (df.___().sum() / ___(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head())

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
Sensor293    91.19
Sensor292    91.19
Sensor157    91.19
Sensor158    91.19
Sensor492    85.58
dtype: float64
```

</details>

### Q5. 결측률 50% 초과 컬럼 제거하기

결측이 너무 많은 컬럼을 통째로 지웁니다.

<details>
<summary><strong>💡 힌트</strong></summary>

'초과'를 뜻하는 비교 연산자, 그리고 열을 지우는 함수입니다.

</details>

In [ ]:
cols_to_drop = miss_pct[miss_pct ___ 50].index.tolist()
df = df.___(columns=cols_to_drop)
print(f"제거한 컬럼: {len(cols_to_drop)}개, 남은 컬럼: {df.shape[1]}개")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
제거한 컬럼: 28개, 남은 컬럼: 564개
```

</details>

### Q6. 남은 결측값을 중앙값으로 채우기

타겟(Pass_Fail)은 목록에서 빼고, 센서 열만 중앙값으로 채웁니다.

<details>
<summary><strong>💡 힌트</strong></summary>

리스트에서 값을 빼는 함수는 `remove()`, 중앙값은 `median()`입니다.

</details>

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.___('Pass_Fail')   # 타겟은 채우지 않습니다

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].___())
print(f"남은 결측: {df.isnull().sum().sum()}")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
남은 결측: 0
```

</details>

### Q7. 분산이 0인 컬럼 제거하기

값이 변하지 않는(분산 0 또는 거의 0) 센서를 지웁니다.

<details>
<summary><strong>💡 힌트</strong></summary>

분산(variance)을 줄인 함수, 그리고 '같다'를 뜻하는 비교 연산자입니다.

</details>

In [ ]:
variances = df[numeric_cols].___()
constant_cols = variances[variances ___ 0].index.tolist()
near_constant = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]
print(f"사용 가능한 센서: {len(numeric_cols)}개")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
사용 가능한 센서: 436개
```

</details>

## 5~6단계 · 특성 선택 · 핵심 변수 EDA

### Q8. 상위 20개 센서 자동 선택하기

불량(+1)을 1로 바꾼 y를 만들고, ANOVA F-검정으로 상위 20개 센서를 고릅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

불량의 값은 1입니다. 상위 K개를 고르는 도구는 `SelectKBest`, 점수 함수는 `f_classif`, 점수는 `scores_`에 저장됩니다.

</details>

In [ ]:
y = (df['Pass_Fail'] == ___).astype(int)

selector = ___(score_func=___, k=20)
selector.fit(df[numeric_cols], y)

f_scores = pd.Series(selector.___, index=numeric_cols).replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=___).head(20).index.tolist()
print("상위 5개 센서:", top_k_cols[:5])

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
상위 5개 센서: ['Sensor59', 'Sensor103', 'Sensor510', 'Sensor348', 'Sensor431']
```

</details>

### Q9. 1위 센서의 정상/불량 분포 비교하기

F-점수 1위 센서를 박스플롯으로 그려 봅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

seaborn의 상자 그래프 함수입니다.

</details>

In [ ]:
plt.figure(figsize=(6, 5))
sns.___(x='Pass_Fail', y=top_k_cols[0], data=df,
            hue='Pass_Fail', palette={-1: '#2ecc71', 1: '#e74c3c'}, legend=False)
plt.title(f'{top_k_cols[0]} — 정상(-1) vs 불량(+1)')
plt.show()

<details>
<summary><strong>✅ 예상 결과</strong></summary>

📊 그래프가 그려지면 성공입니다.

</details>

## 7~9단계 · 전처리 · 모델 학습 · 평가

### Q10. 학습/테스트 데이터 나누기

20%를 테스트로 떼어 내고, 불량 비율이 양쪽에 똑같이 유지되게 나눕니다.

<details>
<summary><strong>💡 힌트</strong></summary>

나누는 함수는 `train_test_split`, 비율 유지 옵션은 '층화'라는 뜻의 이름입니다.

</details>

In [ ]:
X = df[top_k_cols]
X_train, X_test, y_train, y_test = ___(
    X, y,
    test_size=___,
    random_state=42,
    ___=y
)
print(f"학습: {X_train.shape}, 불량 {y_train.sum()}건 / 테스트: {X_test.shape}, 불량 {y_test.sum()}건")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
학습: (1253, 20), 불량 83건 / 테스트: (314, 20), 불량 21건
```

</details>

### Q11. 표준화하기

학습 데이터로만 기준(평균·표준편차)을 정하고, 테스트 데이터는 그 기준으로 바꾸기만 합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

학습 데이터는 맞추고 바꾸기(`fit_transform`), 테스트 데이터는 바꾸기만(`transform`) 합니다.

</details>

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.___(X_train)
X_test_scaled = scaler.___(X_test)

<details>
<summary><strong>✅ 예상 결과</strong></summary>

출력 없이 오류 없이 실행되면 성공입니다.

</details>

### Q12. 로지스틱 회귀 학습하기

적은 쪽(불량)에 가중치를 주고 학습합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

불균형을 자동으로 맞추는 가중치 값은 `'balanced'`, 학습 함수는 `fit()`입니다.

</details>

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000, class_weight=___)
lr_model.___(X_train_scaled, y_train)

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
[31m---------------------------------------------------------------------------[39m
[31mUnicodeDecodeError[39m                        Traceback (most recent call last)
[36mFile [39m[32m~\AppData\Local\Temp\claude\D--Github-Data-WS-kitae-data\79102a29-4cc8-4d0e-8aca-e509527f29ea\scratchpad\venv\Lib\site-packages\sklearn\utils\_repr_html\base.py:163[39m, in [36mReprHTMLMixin._repr_mimebundle_[39m[34m(self, **kwargs)[39m
[32m    161[39m output = {[33m"[39m[33mtext/plain[39m[33m"[39m: [38;5;28mrepr[39m([38;5;28mself[39m)}
[32m    162[39m [38;5;28;01mif[39;00m get_config()[[33m"[39m[33mdisplay[39m[33m"[39m] == [33m"[39m[33mdiagram[39m[33m"[39m:
[32m--> [39m[32m163[39m     output[[33m"[39m[33mtext/html[39m[33m"[39m] = [30;43mself[39;49m[30;43m.[39;49m[30;43m_html_repr[39;49m[30;43m([39;49m[30;43m)[39;49m
[32m    164[39m [38;5;28;01mreturn[39;00m output

[36mFile [39m[32m~\AppData\Local\Temp\claude\D--Github-Data-WS-kitae-data\79102a29-4cc8-4d0e-8aca-e509527f29ea\scratchpad\venv\Lib\site-packages\sklearn\utils\_repr_html\estimator.py:564[39m, in [36mestimator_html_repr[39m[34m(estimator)[39m
[32m    554[39m _write_estimator_html(
[32m    555[39m     out,
[32m    556[39m     estimator,
[32m   (...)[39m[32m    561[39m     is_fitted_icon=is_fitted_icon,
[32m    562[39m )
[32m    563[39m [38;5;28;01mwith[39;00m [38;5;28mopen[39m([38;5;28mstr[39m(Path([34m__file__[39m).parent / [33m"[39m[33mestimator.js[39m[33m"[39m), [33m"[39m[33mr[39m[33m"[39m) [38;5;28;01mas[39;00m f:
[32m--> [39m[32m564[39m     script = [30;43mf[39;49m[30;43m.[39;49m[30;43mread[39;49m[30;43m([39;49m[30;43m)[39;49m
[32m    566[39m html_end = (
[32m    567[39m     [33mf[39m[33m"[39m[33m</div></div><script>[39m[38;5;132;01m{[39;00mscript[38;5;132;01m}[39;00m[33m"[39m
[32m    568[39m     [33mf[39m[33m"[39m[38;5;130;01m\n[39;00m[33mforceTheme([39m[33m'[39m[38;5;132;01m{[39;00mcontainer_id[38;5;132;01m}[39;00m[33m'[39m[33m);</script></body>[39m[33m"[39m
[32m    569[39m )
[32m    571[39m out.write(html_end)

[31mUnicodeDecodeError[39m: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence
[31m---------------------------------------------------------------------------[39m
[31mUnicodeDecodeError[39m                        Traceback (most recent call last)
... (이하 생략)
```

</details>

### Q13. 랜덤 포레스트 학습하기

나무(tree) 200개로 이루어진 숲(forest) 모델을 학습합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

무작위(Random) 숲(Forest) 분류기(Classifier)입니다.

</details>

In [ ]:
rf_model = ___(n_estimators=200, max_depth=8,
                                  random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
[31m---------------------------------------------------------------------------[39m
[31mUnicodeDecodeError[39m                        Traceback (most recent call last)
[36mFile [39m[32m~\AppData\Local\Temp\claude\D--Github-Data-WS-kitae-data\79102a29-4cc8-4d0e-8aca-e509527f29ea\scratchpad\venv\Lib\site-packages\sklearn\utils\_repr_html\base.py:163[39m, in [36mReprHTMLMixin._repr_mimebundle_[39m[34m(self, **kwargs)[39m
[32m    161[39m output = {[33m"[39m[33mtext/plain[39m[33m"[39m: [38;5;28mrepr[39m([38;5;28mself[39m)}
[32m    162[39m [38;5;28;01mif[39;00m get_config()[[33m"[39m[33mdisplay[39m[33m"[39m] == [33m"[39m[33mdiagram[39m[33m"[39m:
[32m--> [39m[32m163[39m     output[[33m"[39m[33mtext/html[39m[33m"[39m] = [30;43mself[39;49m[30;43m.[39;49m[30;43m_html_repr[39;49m[30;43m([39;49m[30;43m)[39;49m
[32m    164[39m [38;5;28;01mreturn[39;00m output

[36mFile [39m[32m~\AppData\Local\Temp\claude\D--Github-Data-WS-kitae-data\79102a29-4cc8-4d0e-8aca-e509527f29ea\scratchpad\venv\Lib\site-packages\sklearn\utils\_repr_html\estimator.py:564[39m, in [36mestimator_html_repr[39m[34m(estimator)[39m
[32m    554[39m _write_estimator_html(
[32m    555[39m     out,
[32m    556[39m     estimator,
[32m   (...)[39m[32m    561[39m     is_fitted_icon=is_fitted_icon,
[32m    562[39m )
[32m    563[39m [38;5;28;01mwith[39;00m [38;5;28mopen[39m([38;5;28mstr[39m(Path([34m__file__[39m).parent / [33m"[39m[33mestimator.js[39m[33m"[39m), [33m"[39m[33mr[39m[33m"[39m) [38;5;28;01mas[39;00m f:
[32m--> [39m[32m564[39m     script = [30;43mf[39;49m[30;43m.[39;49m[30;43mread[39;49m[30;43m([39;49m[30;43m)[39;49m
[32m    566[39m html_end = (
[32m    567[39m     [33mf[39m[33m"[39m[33m</div></div><script>[39m[38;5;132;01m{[39;00mscript[38;5;132;01m}[39;00m[33m"[39m
[32m    568[39m     [33mf[39m[33m"[39m[38;5;130;01m\n[39;00m[33mforceTheme([39m[33m'[39m[38;5;132;01m{[39;00mcontainer_id[38;5;132;01m}[39;00m[33m'[39m[33m);</script></body>[39m[33m"[39m
[32m    569[39m )
[32m    571[39m out.write(html_end)

[31mUnicodeDecodeError[39m: 'cp949' codec can't decode byte 0xe2 in position 2950: illegal multibyte sequence
[31m---------------------------------------------------------------------------[39m
[31mUnicodeDecodeError[39m                        Traceback (most recent call last)
... (이하 생략)
```

</details>

### Q14. 혼동행렬과 분류 리포트 보기

로지스틱 회귀의 예측을 혼동행렬과 리포트로 평가합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

예측 함수는 `predict()`, 혼동행렬 함수는 `confusion_matrix()`입니다.

</details>

In [ ]:
y_pred_lr = lr_model.___(X_test_scaled)
print(___(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=['정상(0)', '불량(1)']))

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
[[221  72]
 [  6  15]]
              precision    recall  f1-score   support

       정상(0)       0.97      0.75      0.85       293
       불량(1)       0.17      0.71      0.28        21

    accuracy                           0.75       314
   macro avg       0.57      0.73      0.56       314
weighted avg       0.92      0.75      0.81       314
```

</details>

### Q15. ROC 곡선의 AUC 구하기

불량일 확률로 ROC 곡선을 만들고 곡선 아래 면적(AUC)을 구합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

확률 예측은 `predict_proba()`, 곡선은 `roc_curve()`, 면적은 `auc()`입니다.

</details>

In [ ]:
y_prob_lr = lr_model.___(X_test_scaled)[:, 1]
fpr, tpr, _ = ___(y_test, y_prob_lr)
print(f"로지스틱 회귀 AUC: {___(fpr, tpr):.3f}")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
로지스틱 회귀 AUC: 0.742
```

</details>

### Q16. 특성 중요도 보기

랜덤 포레스트가 중요하게 쓴 센서 상위 5개를 봅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

특성(feature) 중요도(importances)가 저장된 속성입니다. 끝에 밑줄 `_`이 붙습니다.

</details>

In [ ]:
importance = pd.Series(rf_model.___, index=top_k_cols)
print(importance.sort_values(ascending=False).head().round(3))

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
Sensor103    0.104
Sensor59     0.097
Sensor21     0.061
Sensor510    0.060
Sensor28     0.056
dtype: float64
```

</details>

## 10 · 보강 실습

### Q17. '항상 정상' 기준 모델과 비교하기

모두 정상(0)이라고 예측하면 정확도와 불량 Recall이 어떻게 되는지 봅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

0으로 가득 찬 배열을 만드는 numpy 함수입니다.

</details>

In [ ]:
always_normal = np.___(len(y_test), dtype=int)
print(f"정확도: {accuracy_score(y_test, always_normal):.3f}")
print(f"불량 Recall: {recall_score(y_test, always_normal, zero_division=0):.3f}")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
정확도: 0.933
불량 Recall: 0.000
```

</details>

### Q18. 혼동행렬 숫자로 Recall 직접 계산하기

혼동행렬을 네 값으로 나누고 불량 Recall = TP / (TP + FN)을 계산합니다.

<details>
<summary><strong>💡 힌트</strong></summary>

2×2 행렬을 한 줄로 펴는 함수는 `ravel()`입니다. 분모에는 놓친 불량(FN)이 들어갑니다.

</details>

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr).___()
manual_recall = tp / (tp + ___)
print(f"TN {tn}, FP {fp}, FN {fn}, TP {tp}")
print(f"불량 Recall: {manual_recall:.3f}")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
TN 221, FP 72, FN 6, TP 15
불량 Recall: 0.714
```

</details>

### Q19. 임계값을 낮춰 보기

불량 확률이 0.3 이상이면 불량으로 판정했을 때 Recall을 봅니다.

<details>
<summary><strong>💡 힌트</strong></summary>

'이상'을 뜻하는 비교 연산자입니다.

</details>

In [ ]:
pred_03 = (y_prob_lr ___ 0.3).astype(int)
print(f"임계값 0.3 불량 Recall: {recall_score(y_test, pred_03):.3f}")
print(f"임계값 0.5 불량 Recall: {recall_score(y_test, y_pred_lr):.3f}")

<details>
<summary><strong>✅ 예상 결과</strong></summary>

```
임계값 0.3 불량 Recall: 0.857
임계값 0.5 불량 Recall: 0.714
```

</details>

## 마무리

모든 빈칸을 채워 끝까지 실행했다면 성공입니다. 틀린 빈칸은 강의 페이지의 해당 단계 설명을 다시 읽고, 빈칸 없이 처음부터 직접 입력해 보세요.